# GLP-1-Related Obesity Clinical Trials and FDA Drug Products

**Milestone 2**


**Author:** Ben Plaisance
**Course:** CHIP 705



### **Research Question:** 
How are clinical trials involving selected GLP-1-related medications distributed across study characteristics, and how do their interventions correspond to FDA-listed drug products?

### Required Files
- `ctg-studies.csv` — ClinicalTrials.gov study data
- `Products.txt` — FDA drug product data
- Python libraries: `pandas` and `matplotlib`

### Project Overview
This project uses Python to analyze GLP-1-related obesity clinical trials and their corresponding FDA drug product records. The code imports and cleans the datasets, identifies five selected GLP-1-related medications, and matches clinical trial interventions to FDA products based on active ingredients. Descriptive statistics and visualizations are generated to examine study characteristics, medication representation, FDA product matches, and trends over time.

In [1]:
# Import important libraries for file managing, text matching, data analysis, and plot creation.
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() / "GLP-1-Related-Trial-and-FDA-Product-Analysis-main"

def find_input(filename): 
    for path in (ROOT / 'data' / filename, ROOT / filename):
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Cannot find {filename}. Put it in the data/ folder or next to the notebook. "
        f"Current working folder: {ROOT}" )

TRIALS_FILE = find_input('ctg-studies.csv')
FDA_FILE = find_input('Products.txt')
OUTPUT = ROOT / 'results'
OUTPUT.mkdir(parents=True, exist_ok=True)
INGREDIENTS = ['SEMAGLUTIDE', 'LIRAGLUTIDE', 'TIRZEPATIDE', 'EXENATIDE', 'DULAGLUTIDE']
AS_OF = pd.Timestamp('2026-11-21') # Cutoff date for snapshot of data, needs to be updated if using later export

FileNotFoundError: Cannot find ctg-studies.csv. Put it in the data/ folder or next to the notebook. Current working folder: C:\Users\benpl\CHIP 705\Milestone 2\GLP-1-Related-Trial-and-FDA-Product-Analysis-main

In [ ]:
#Reads original files, save Ids such as application/product number, replace missing text with empty strings
# Removes rows without Ids and drops any repeat identifiers in study/product
trials = pd.read_csv(TRIALS_FILE, dtype=str).fillna('')
fda = pd.read_csv(FDA_FILE, sep='\t', dtype=str, encoding='latin1').fillna('')
trials = trials.loc[trials['NCT Number'].str.strip().ne('')].drop_duplicates('NCT Number').copy()
fda = fda.loc[fda['ApplNo'].str.strip().ne('') &
fda['ProductNo'].str.strip().ne('')].drop_duplicates(['ApplNo', 'ProductNo']).copy()
print(f'Trial records: {len(trials):,}; unique FDA product records: {len(fda):,}')

In [ ]:
#Counts number of trials in each status and phase category, any blank fields are saved as 'Not reported' instead of being removed
#Exports summary tables for the report and creates two bar charts showing the counts for each phase/status
status = trials['Study Status'].replace('', 'Not reported').value_counts().rename_axis('Status').reset_index(name='Studies')
phase = trials['Phases'].replace('', 'Not reported').value_counts().rename_axis('Phase').reset_index(name='Studies')
status.to_csv(OUTPUT / 'study_status.csv', index=False)
phase.to_csv(OUTPUT / 'study_phase.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15,6))
for ax, df, category, title in [(axes[0], status, 'Status', 'Study status'), (axes[1], phase, 'Phase', 'Study phase')]:
    d = df.sort_values('Studies')
    ax.barh(d[category], d['Studies'])
    ax.set_title(title)
    ax.set_xlabel('Number of distinct studies')
fig.suptitle('ClinicalTrials.gov obesity / GLP-1 search results')
fig.tight_layout()
fig.savefig(OUTPUT / '01__study_landscape.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# Split clinical trial interventions into separate records, and separate type from name.
#Match only the 5 selected ingredients by name, ignoring placebo interventions and removing duplicate records
#Use unique NCT numbers to count studies to avoid double counting, create figure representing ingredient counts
records = []
for _, row in trials.iterrows():
    for raw in row['Interventions'].split('|'):
        raw = raw.strip()
        if not raw or ':' not in raw:
            continue
        kind, name = raw.split(':', 1)
        name = name.strip()

        ingredient = next((x for x in INGREDIENTS if re.search(r'\b' + x + r'\b', name.upper())), None)
        if 'PLACEBO' in name.upper():
            ingredient = None
        records.append({'NCT Number': row['NCT Number'], 'Intervention type': kind.strip().upper(), 'Intervention name': name, 'Ingredient': ingredient})
interventions = pd.DataFrame(records).drop_duplicates()
selected = interventions.dropna(subset=['Ingredient']).copy()
by_ingredient = selected.groupby('Ingredient').agg(Studies=('NCT Number', 'nunique'), Interventions=('Intervention name', 'count')).reset_index().sort_values('Studies', ascending=False)
by_ingredient.to_csv(OUTPUT / 'ingredient_studies.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
d = by_ingredient.sort_values('Studies')
ax.barh(d['Ingredient'], d['Studies'])
ax.set_title('Clinical trials naming selected GLP-1 related ingredients')
ax.set_xlabel('Distinct studies (one study may appear under multiple ingredients)')
fig.tight_layout()
fig.savefig(OUTPUT / '02_studies_by_ingredient.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
#Checks FDA  ingredient list for each selected ingredient and counts unique products using the (ApplNo, ProductNo) combination
#Counts distinct FDA application numbers and merges FDA product counts with above clinical trial ingredient counts
#Saves table and chart, creates table showing the FDA product record counts for each ingredient
product_records = []
for ingredient in INGREDIENTS:
    mask = fda['ActiveIngredient'].str.upper().str.contains(r'\b' + ingredient + r'\b', regex=True, na=False)
    subset = fda.loc[mask, ['ApplNo', 'ProductNo']].drop_duplicates()
    product_records.append({'Ingredient': ingredient, 'FDA product records': len(subset), 'FDA applications': subset['ApplNo'].nunique()})
product_counts = pd.DataFrame(product_records)
comparison = by_ingredient.merge(product_counts, on='Ingredient', how='left')
comparison.to_csv(OUTPUT / 'ingredient_fda_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
d = comparison.sort_values('FDA product records')
ax.barh(d['Ingredient'], d['FDA product records'])
ax.set_title('FDA product records listing each selected active ingredient')
ax.set_xlabel('DIstinct FDA product records (application + product number)')
fig.tight_layout()
fig.savefig(OUTPUT / '03_fda_product_records.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# Parse through study start dates, covering all format types
#Excludes dates after the snapshot cutoff to avoid any future planned studies, including only those that have commenced
#Grouped by year, then a figure is created that shows the distinct studies per year from 2000 to cutoff date
trials['Start date parsed'] = pd.to_datetime(trials['Start Date'], errors='coerce', format='mixed')
historical = trials.loc[trials['Start date parsed'].notna() & trials['Start date parsed'].le(AS_OF)].copy()
historical['Year'] = historical['Start date parsed'].dt.year
years = historical.groupby('Year')['NCT Number'].nunique().reset_index(name='Studies')
years.to_csv(OUTPUT / 'start_years.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(years['Year'], years['Studies'])
ax.set_title(f'Study start years through {AS_OF.date()}')
ax.set_xlabel('Start year')
ax.set_ylabel('Distinct studies')
fig.tight_layout()
fig.savefig(OUTPUT / '04_start_years.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
#Whole bunch of summary statements to be used in the report and in understanding the data
# Reports the overall study count total, common statuses, and ingredient coverage
matched_studies = selected['NCT Number'].nunique()
print('\nRESULTS SUMMARY')
print(f'Total distinct studies: {len(trials):,}')
print(f'Completed: {(trials["Study Status"] == "COMPLETED").sum():,}')
print(f'Recruiting: {(trials["Study Status"] == "RECRUITING").sum():,}')
print(f'Studies naming at least one selected ingredient: {matched_studies:,}')
print(f'Studies not naming a selected ingredient: {len(trials) - matched_studies:,}')
print(f'Future-dated start records excluded from trend:{(trials["Start date parsed"] > AS_OF).sum():,}')
print('\nIngredient and FDA listing comparison:')
print(comparison.to_string(index=False))
#Show the saved location of all files to ensure they went to the right place.
print(f'n\Figures and CSV summaries saved to: {OUTPUT}')